In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
from scipy.linalg import expm
import os
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#FAFAFA',
    'axes.edgecolor': '#CCCCCC',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': '#CCCCCC',
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
})

C_RVNN  = '#E74C3C'
C_CVNN  = '#2E86C1'
C_IDEAL = '#27AE60'

In [2]:
def generate_larmor_data(samples=2000, t_max=10):
    """Larmor precession: spin-1/2 in uniform B field."""
    print(f"  [data] Generating {samples} Larmor precession snapshots...")
    t = np.linspace(0, t_max, samples)
    omega = 2.0
    alpha = np.cos(omega * t / 2)
    beta  = -1j * np.sin(omega * t / 2)
    y_complex = np.stack([alpha, beta], axis=1).astype(np.complex64)
    y_real = np.stack([alpha.real, alpha.imag,
                       beta.real, beta.imag], axis=1).astype(np.float32)
    X = t.reshape(-1, 1).astype(np.float32)
    return X, y_real, y_complex


def generate_damped_precession_data(samples=2000, t_max=10):
    """Damped precession: T1-like amplitude decay + rotation."""
    print(f"  [data] Generating {samples} damped precession snapshots...")
    t = np.linspace(0, t_max, samples)
    omega, gamma = 2.0, 0.3
    alpha = np.exp(-gamma * t) * np.cos(omega * t / 2)
    beta  = -1j * np.exp(-gamma * t) * np.sin(omega * t / 2)
    norm  = np.sqrt(np.abs(alpha)**2 + np.abs(beta)**2) + 1e-8
    alpha, beta = alpha / norm, beta / norm
    y_complex = np.stack([alpha, beta], axis=1).astype(np.complex64)
    y_real = np.stack([alpha.real, alpha.imag,
                       beta.real, beta.imag], axis=1).astype(np.float32)
    X = t.reshape(-1, 1).astype(np.float32)
    return X, y_real, y_complex


def generate_two_qubit_data(samples=2000, t_max=10):
    """Two-qubit entangled system: Ising model with transverse field.

    Hamiltonian: H = J * sigma_z x sigma_z + B * (sigma_x x I + I x sigma_x)
    The ZZ coupling creates genuine entanglement from the product initial
    state |00>.  This is qualitatively harder than a product-state evolution
    because the state cannot be factored at generic times.
    """
    print(f"  [data] Generating {samples} two-qubit snapshots...")
    t = np.linspace(0, t_max, samples)

    sx = np.array([[0, 1], [1, 0]], dtype=complex)
    sz = np.array([[1, 0], [0, -1]], dtype=complex)
    I2 = np.eye(2, dtype=complex)

    J, B = 1.0, 0.5
    H = J * np.kron(sz, sz) + B * (np.kron(sx, I2) + np.kron(I2, sx))

    psi0 = np.array([1, 0, 0, 0], dtype=complex)  # |00>

    y_complex = np.zeros((samples, 4), dtype=np.complex64)
    for i, ti in enumerate(t):
        U = expm(-1j * H * ti)
        y_complex[i] = (U @ psi0).astype(np.complex64)

    y_real = np.stack([
        y_complex[:, 0].real, y_complex[:, 0].imag,
        y_complex[:, 1].real, y_complex[:, 1].imag,
        y_complex[:, 2].real, y_complex[:, 2].imag,
        y_complex[:, 3].real, y_complex[:, 3].imag,
    ], axis=1).astype(np.float32)
    X = t.reshape(-1, 1).astype(np.float32)
    return X, y_real, y_complex

In [3]:
class BaselineRVNN(nn.Module):
    """Standard RVNN — no quantum-aware constraints.
    Output is unconstrained in R^n; norm != 1 in general."""
    def __init__(self, out_features=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, out_features),
        )

    def forward(self, x):
        return self.net(x)


def _fair_hidden(target_params, out_features):
    """Find hidden size that gives ~target_params for a 3-layer RVNN.
    params = hidden^2 + 3*hidden + out_features*(hidden+1)
    """
    a = 1
    b = 3 + out_features
    c = out_features - target_params
    return max(int((-b + np.sqrt(b**2 - 4*a*c)) / (2*a)), 1)


class FairRVNN(nn.Module):
    """Parameter-matched RVNN: same param count as CVNN.
    This isolates the effect of complex arithmetic from raw capacity."""
    def __init__(self, out_features=4, hidden=None):
        super().__init__()
        if hidden is None:
            hidden = _fair_hidden(count_params(QuantumCVNN(out_features // 2)),
                                  out_features)
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_features),
        )

    def forward(self, x):
        return self.net(x)


class ComplexLinear(nn.Module):
    """Fully complex linear layer: W in C^{out x in}.

    Implements (W_re + i W_im)(z_re + i z_im) which naturally couples
    amplitude and phase — unlike RVNN which treats Re/Im independently.
    """
    def __init__(self, in_features, out_features):
        super().__init__()
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W_re = nn.Parameter(torch.randn(out_features, in_features) * scale)
        self.W_im = nn.Parameter(torch.randn(out_features, in_features) * scale)
        self.b_re = nn.Parameter(torch.zeros(out_features))
        self.b_im = nn.Parameter(torch.zeros(out_features))

    def forward(self, z):
        out_re = z.real @ self.W_re.T - z.imag @ self.W_im.T + self.b_re
        out_im = z.real @ self.W_im.T + z.imag @ self.W_re.T + self.b_im
        return torch.complex(out_re, out_im)


def complex_cardioid(z):
    """Cardioid activation: 0.5 * (1 + cos(angle(z))) * z

    Phase-aware and smooth — preserves phase information while providing
    a nonlinearity. Superior to split-ReLU for quantum applications.
    """
    phase = torch.angle(z)
    return 0.5 * (1.0 + torch.cos(phase)) * z


class QuantumCVNN(nn.Module):
    """Complex-Valued NN with enforced quantum constraints.

    Constraints:
      1. C-linearity: ComplexLinear couples amplitude & phase
      2. Phase-aware activation: cardioid preserves phase structure
      3. Norm conservation: output projected to ||psi|| = 1
    """
    def __init__(self, out_features=2):
        super().__init__()
        self.fc1 = ComplexLinear(1, 64)
        self.fc2 = ComplexLinear(64, 64)
        self.fc3 = ComplexLinear(64, out_features)

    def forward(self, z):
        z = complex_cardioid(self.fc1(z))
        z = complex_cardioid(self.fc2(z))
        z = self.fc3(z)
        # CONSTRAINT: project onto unit sphere  ||psi|| = 1
        norm = torch.sqrt(torch.sum(torch.abs(z)**2, dim=1, keepdim=True) + 1e-8)
        return z / norm

In [4]:
def fidelity_loss(psi_pred, psi_true):
    """L = 1 - |<psi_true|psi_pred>|^2

    Properties:
      - Phase-invariant: |psi> and e^{i theta}|psi> give same loss
      - L = 0 for identical states, L = 1 for orthogonal states
      - Directly optimises the quantum-mechanical overlap
    """
    inner = torch.sum(torch.conj(psi_true) * psi_pred, dim=1)
    return torch.mean(1.0 - torch.abs(inner)**2)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def to_complex_input(X):
    t = torch.tensor(X).float()
    return torch.complex(t, torch.zeros_like(t))


def rvnn_to_complex(preds, n_components):
    parts = []
    for i in range(n_components):
        parts.append(preds[:, 2*i] + 1j * preds[:, 2*i+1])
    return np.stack(parts, axis=1)


def normalize_complex(y):
    norms = np.sqrt(np.sum(np.abs(y)**2, axis=1, keepdims=True)) + 1e-8
    return y / norms


def compute_fidelity(y_true, y_pred):
    dot = np.sum(np.conj(y_true) * y_pred, axis=1)
    return np.abs(dot)**2


def compute_norm(y):
    return np.sqrt(np.sum(np.abs(y)**2, axis=1))


def compute_phase_error(y_true, y_pred):
    """Phase error with global phase alignment.

    Quantum states |psi> and e^{i*theta}|psi> are physically identical
    (global phase freedom).  Before computing component-wise phase
    differences we align the global phase by rotating y_pred so that
    <y_true | y_pred> becomes real and positive.
    """
    inner = np.sum(np.conj(y_true) * y_pred, axis=1, keepdims=True)
    phase_align = np.exp(-1j * np.angle(inner))
    y_pred_aligned = y_pred * phase_align

    diff = np.angle(y_true) - np.angle(y_pred_aligned)
    diff = np.mod(diff + np.pi, 2 * np.pi) - np.pi
    return np.mean(np.abs(diff), axis=1)


def train_rvnn(model, X_t, y_t, epochs=1500, lr=0.005):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for ep in range(epochs):
        optimizer.zero_grad()
        loss = nn.MSELoss()(model(X_t), y_t)
        loss.backward()
        optimizer.step()
        if ep % 10 == 0:
            losses.append(loss.item())
    return losses


def train_cvnn(model, X_c, y_c, epochs=1500, lr=0.005):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for ep in range(epochs):
        optimizer.zero_grad()
        loss = fidelity_loss(model(X_c), y_c)
        loss.backward()
        optimizer.step()
        if ep % 10 == 0:
            losses.append(loss.item())
    return losses

In [5]:
EPOCHS = 1500
LR     = 0.005
N_TRIALS = 10

DATASETS = {
    "Larmor":    generate_larmor_data,
    "Damped":    generate_damped_precession_data,
    "Two-Qubit": generate_two_qubit_data,
}

print("=" * 70)
print("  QUANTUM STATE PREDICTION: CVNN vs RVNN")
print("  with Enforced Mathematical Constraints")
print("=" * 70)

# ── Parameter counts ─────────────────────────────────────────────────────────
n_rvnn = count_params(BaselineRVNN(4))
n_cvnn = count_params(QuantumCVNN(2))
print(f"\n  Model parameters:  RVNN = {n_rvnn:,}   CVNN = {n_cvnn:,}   ratio = {n_cvnn/n_rvnn:.2f}")

# ── Benchmark across datasets ────────────────────────────────────────────────
all_results   = {}
all_losses    = {}
all_preds     = {}

for ds_name, ds_fn in DATASETS.items():
    print(f"\n{'─'*60}\n  Dataset: {ds_name}\n{'─'*60}")
    X, y_real, y_cplx = ds_fn()
    n_r = y_real.shape[1]
    n_c = y_cplx.shape[1]

    X_t  = torch.tensor(X)
    yr_t = torch.tensor(y_real)
    X_c  = to_complex_input(X)
    yc_t = torch.tensor(y_cplx).to(torch.complex64)

    # Train RVNN
    torch.manual_seed(42)
    model_r = BaselineRVNN(n_r)
    print(f"  Training RVNN ({count_params(model_r):,} params, MSE loss)...")
    loss_r = train_rvnn(model_r, X_t, yr_t, EPOCHS, LR)

    # Train CVNN
    torch.manual_seed(42)
    model_c = QuantumCVNN(n_c)
    print(f"  Training CVNN ({count_params(model_c):,} params, fidelity loss)...")
    loss_c = train_cvnn(model_c, X_c, yc_t, EPOCHS, LR)

    # Evaluate
    with torch.no_grad():
        pr_raw  = rvnn_to_complex(model_r(X_t).numpy(), n_c)
        pr_norm = normalize_complex(pr_raw)
        pc      = model_c(X_c).numpy()

    fid_r   = compute_fidelity(y_cplx, pr_norm)
    fid_c   = compute_fidelity(y_cplx, pc)
    norm_r  = compute_norm(pr_raw)
    norm_c  = compute_norm(pc)
    phase_r = compute_phase_error(y_cplx, pr_norm)
    phase_c = compute_phase_error(y_cplx, pc)

    all_results[ds_name] = dict(
        fid_rvnn=np.mean(fid_r), fid_cvnn=np.mean(fid_c),
        fid_rvnn_s=fid_r, fid_cvnn_s=fid_c,
        norm_r_mean=np.mean(norm_r), norm_r_std=np.std(norm_r),
        norm_c_mean=np.mean(norm_c), norm_c_std=np.std(norm_c),
        norm_r_s=norm_r, norm_c_s=norm_c,
        phase_r=np.mean(phase_r), phase_c=np.mean(phase_c),
        phase_r_s=phase_r, phase_c_s=phase_c,
    )
    all_losses[ds_name] = dict(rvnn=loss_r, cvnn=loss_c)
    all_preds[ds_name]  = dict(X=X, y=y_cplx, rvnn=pr_norm, cvnn=pc, rvnn_raw=pr_raw)

    print(f"  {'Metric':<28} {'RVNN':>10} {'CVNN':>10}")
    print(f"  {'─'*48}")
    print(f"  {'Fidelity (normalized)':<28} {np.mean(fid_r):>10.6f} {np.mean(fid_c):>10.6f}")
    print(f"  {'Norm (mean +/- std)':<28}  {np.mean(norm_r):.3f}+/-{np.std(norm_r):.3f}  {np.mean(norm_c):.3f}+/-{np.std(norm_c):.3f}")
    print(f"  {'Phase Error (rad)':<28} {np.mean(phase_r):>10.4f} {np.mean(phase_c):>10.4f}")

# ── Statistical analysis (10 trials, all datasets) ───────────────────────────
print(f"\n{'='*70}")
print(f"  STATISTICAL ANALYSIS: {N_TRIALS} Independent Trials")
print(f"{'='*70}")

stat_results = {}
for ds_name, ds_fn in DATASETS.items():
    X_s, yr_s, yc_s = ds_fn()
    n_r = yr_s.shape[1]; n_c = yc_s.shape[1]
    X_st = torch.tensor(X_s); yr_st = torch.tensor(yr_s)
    X_sc = to_complex_input(X_s); yc_st = torch.tensor(yc_s).to(torch.complex64)

    scores_r, scores_c = [], []
    print(f"\n  Dataset: {ds_name}")
    for seed in range(N_TRIALS):
        torch.manual_seed(seed); np.random.seed(seed)

        mr = BaselineRVNN(n_r)
        train_rvnn(mr, X_st, yr_st, 1000, LR)
        with torch.no_grad():
            pr = normalize_complex(rvnn_to_complex(mr(X_st).numpy(), n_c))
        scores_r.append(np.mean(compute_fidelity(yc_s, pr)))

        mc = QuantumCVNN(n_c)
        train_cvnn(mc, X_sc, yc_st, 1000, LR)
        with torch.no_grad():
            pc = mc(X_sc).numpy()
        scores_c.append(np.mean(compute_fidelity(yc_s, pc)))

        print(f"    Trial {seed+1:2d} | RVNN: {scores_r[-1]:.4f}  CVNN: {scores_c[-1]:.4f}")

    t_s, p_v = stats.ttest_ind(scores_c, scores_r)
    stat_results[ds_name] = dict(rvnn=scores_r, cvnn=scores_c, t=t_s, p=p_v)
    sig = "Significant" if p_v < 0.05 else "Not significant"
    print(f"    RVNN: {np.mean(scores_r):.4f} +/- {np.std(scores_r):.4f}")
    print(f"    CVNN: {np.mean(scores_c):.4f} +/- {np.std(scores_c):.4f}")
    print(f"    t={t_s:.4f}  p={p_v:.6f}  {sig}")

# ── Extrapolation test ────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("  EXTRAPOLATION TEST (train t in [0,8], test t in [8,12])")
print(f"{'='*70}")

X_ext, yr_ext, yc_ext = generate_larmor_data(2400, 12)
split = int((8.0 / 12.0) * len(X_ext))

X_tr_t  = torch.tensor(X_ext[:split])
yr_tr_t = torch.tensor(yr_ext[:split])
Xc_tr   = to_complex_input(X_ext[:split])
yc_tr_t = torch.tensor(yc_ext[:split]).to(torch.complex64)

torch.manual_seed(42)
mr_ext = BaselineRVNN(4); train_rvnn(mr_ext, X_tr_t, yr_tr_t, 1500, LR)
torch.manual_seed(42)
mc_ext = QuantumCVNN(2);  train_cvnn(mc_ext, Xc_tr, yc_tr_t, 1500, LR)

with torch.no_grad():
    pr_ext = normalize_complex(rvnn_to_complex(mr_ext(torch.tensor(X_ext)).numpy(), 2))
    pc_ext = mc_ext(to_complex_input(X_ext)).numpy()

fid_r_ext = compute_fidelity(yc_ext, pr_ext)
fid_c_ext = compute_fidelity(yc_ext, pc_ext)

print(f"  RVNN — train: {np.mean(fid_r_ext[:split]):.4f} | extrap: {np.mean(fid_r_ext[split:]):.4f}")
print(f"  CVNN — train: {np.mean(fid_c_ext[:split]):.4f} | extrap: {np.mean(fid_c_ext[split:]):.4f}")


# =============================================================================
# SECTION 4b: FAIR PARAMETER-MATCHED COMPARISON
# =============================================================================

print(f"\n{'='*70}")
print("  FAIR COMPARISON: Parameter-Matched RVNN vs CVNN")
print(f"{'='*70}")

fair_results = {}
fair_stat_results = {}

for ds_name, ds_fn in DATASETS.items():
    X_f, yr_f, yc_f = ds_fn()
    n_r = yr_f.shape[1]; n_c = yc_f.shape[1]
    X_ft = torch.tensor(X_f); yr_ft = torch.tensor(yr_f)
    X_fc = to_complex_input(X_f); yc_ft = torch.tensor(yc_f).to(torch.complex64)

    torch.manual_seed(42)
    mf = FairRVNN(n_r)
    n_fair = count_params(mf)
    n_cvnn_ds = count_params(QuantumCVNN(n_c))
    print(f"\n  Dataset: {ds_name}  (FairRVNN={n_fair:,}  CVNN={n_cvnn_ds:,} params)")
    print(f"  Training FairRVNN ({n_fair:,} params, MSE loss)...")
    train_rvnn(mf, X_ft, yr_ft, EPOCHS, LR)

    torch.manual_seed(42)
    mc = QuantumCVNN(n_c)
    print(f"  Training CVNN ({n_cvnn_ds:,} params, fidelity loss)...")
    train_cvnn(mc, X_fc, yc_ft, EPOCHS, LR)

    with torch.no_grad():
        pf_raw = rvnn_to_complex(mf(X_ft).numpy(), n_c)
        pf_norm = normalize_complex(pf_raw)
        pc = mc(X_fc).numpy()

    fid_f = compute_fidelity(yc_f, pf_norm)
    fid_c = compute_fidelity(yc_f, pc)
    fair_results[ds_name] = dict(fid_fair=np.mean(fid_f), fid_cvnn=np.mean(fid_c),
                                  n_fair=n_fair, n_cvnn=n_cvnn_ds)
    print(f"  {'Metric':<28} {'FairRVNN':>10} {'CVNN':>10}")
    print(f"  {'─'*48}")
    print(f"  {'Fidelity (normalized)':<28} {np.mean(fid_f):>10.6f} {np.mean(fid_c):>10.6f}")

    # 10-trial statistics
    scores_f, scores_c = [], []
    for seed in range(N_TRIALS):
        torch.manual_seed(seed); np.random.seed(seed)

        mf_t = FairRVNN(n_r)
        train_rvnn(mf_t, X_ft, yr_ft, 1000, LR)
        with torch.no_grad():
            pf_t = normalize_complex(rvnn_to_complex(mf_t(X_ft).numpy(), n_c))
        scores_f.append(np.mean(compute_fidelity(yc_f, pf_t)))

        mc_t = QuantumCVNN(n_c)
        train_cvnn(mc_t, X_fc, yc_ft, 1000, LR)
        with torch.no_grad():
            pc_t = mc_t(X_fc).numpy()
        scores_c.append(np.mean(compute_fidelity(yc_f, pc_t)))

        print(f"    Trial {seed+1:2d} | FairRVNN: {scores_f[-1]:.4f}  CVNN: {scores_c[-1]:.4f}")

    t_s, p_v = stats.ttest_ind(scores_c, scores_f)
    fair_stat_results[ds_name] = dict(fair=scores_f, cvnn=scores_c, t=t_s, p=p_v)
    sig = "Significant" if p_v < 0.05 else "Not significant"
    print(f"    FairRVNN: {np.mean(scores_f):.4f} +/- {np.std(scores_f):.4f}")
    print(f"    CVNN:     {np.mean(scores_c):.4f} +/- {np.std(scores_c):.4f}")
    print(f"    t={t_s:.4f}  p={p_v:.6f}  {sig}")


# =============================================================================
# SECTION 4c: PHYSICAL VALIDITY SCORE
# =============================================================================

print(f"\n{'='*70}")
print("  PHYSICAL VALIDITY: Born Rule Compliance")
print(f"{'='*70}")
print("  (% of predictions with ||ψ|| ∈ [0.99, 1.01])")

validity_results = {}
for ds_name in DATASETS:
    P = all_preds[ds_name]
    # Raw RVNN output (before manual normalization)
    norm_r_raw = compute_norm(P['rvnn_raw'])
    norm_c = compute_norm(P['cvnn'])
    valid_r = 100.0 * np.mean((norm_r_raw >= 0.99) & (norm_r_raw <= 1.01))
    valid_c = 100.0 * np.mean((norm_c >= 0.99) & (norm_c <= 1.01))
    validity_results[ds_name] = dict(rvnn=valid_r, cvnn=valid_c)
    print(f"  {ds_name:<12}  RVNN: {valid_r:6.1f}%   CVNN: {valid_c:6.1f}%")


# =============================================================================
# SECTION 4d: NOISE ROBUSTNESS TEST
# =============================================================================

print(f"\n{'='*70}")
print("  NOISE ROBUSTNESS TEST")
print(f"{'='*70}")
print("  (Train on clean data, test with Gaussian noise on inputs)")

noise_levels = [0.0, 0.01, 0.05, 0.1, 0.2, 0.5]
noise_results = {}

# Use Larmor dataset for this test
X_n, yr_n, yc_n = generate_larmor_data()
n_r_n = yr_n.shape[1]; n_c_n = yc_n.shape[1]
X_nt = torch.tensor(X_n); yr_nt = torch.tensor(yr_n)
X_nc = to_complex_input(X_n); yc_nt = torch.tensor(yc_n).to(torch.complex64)

# Train on clean data
torch.manual_seed(42)
mr_noise = BaselineRVNN(n_r_n)
train_rvnn(mr_noise, X_nt, yr_nt, EPOCHS, LR)

torch.manual_seed(42)
mf_noise = FairRVNN(n_r_n)
train_rvnn(mf_noise, X_nt, yr_nt, EPOCHS, LR)

torch.manual_seed(42)
mc_noise = QuantumCVNN(n_c_n)
train_cvnn(mc_noise, X_nc, yc_nt, EPOCHS, LR)

fid_rvnn_noise, fid_fair_noise, fid_cvnn_noise = [], [], []
norm_rvnn_noise, norm_fair_noise = [], []

for sigma in noise_levels:
    # Add noise to inputs
    np.random.seed(0)
    X_noisy = X_n + np.random.normal(0, sigma, X_n.shape).astype(np.float32)
    X_noisy_t = torch.tensor(X_noisy)
    X_noisy_c = to_complex_input(X_noisy)

    # Regenerate true states for noisy inputs (ground truth at noisy time points)
    omega = 2.0
    t_noisy = X_noisy.flatten()
    alpha_n = np.cos(omega * t_noisy / 2)
    beta_n  = -1j * np.sin(omega * t_noisy / 2)
    yc_noisy = np.stack([alpha_n, beta_n], axis=1).astype(np.complex64)

    with torch.no_grad():
        pr_n_raw = rvnn_to_complex(mr_noise(X_noisy_t).numpy(), n_c_n)
        pr_n = normalize_complex(pr_n_raw)
        pf_n_raw = rvnn_to_complex(mf_noise(X_noisy_t).numpy(), n_c_n)
        pf_n = normalize_complex(pf_n_raw)
        pc_n = mc_noise(X_noisy_c).numpy()

    fid_rvnn_noise.append(np.mean(compute_fidelity(yc_noisy, pr_n)))
    fid_fair_noise.append(np.mean(compute_fidelity(yc_noisy, pf_n)))
    fid_cvnn_noise.append(np.mean(compute_fidelity(yc_noisy, pc_n)))
    norm_rvnn_noise.append(np.std(compute_norm(pr_n_raw)))
    norm_fair_noise.append(np.std(compute_norm(pf_n_raw)))

    print(f"  σ={sigma:.2f}  RVNN: {fid_rvnn_noise[-1]:.4f}  "
          f"FairRVNN: {fid_fair_noise[-1]:.4f}  CVNN: {fid_cvnn_noise[-1]:.4f}")

noise_results = dict(sigma=noise_levels,
                     rvnn=fid_rvnn_noise, fair=fid_fair_noise,
                     cvnn=fid_cvnn_noise,
                     norm_rvnn=norm_rvnn_noise, norm_fair=norm_fair_noise)

  QUANTUM STATE PREDICTION: CVNN vs RVNN
  with Enforced Mathematical Constraints

  Model parameters:  RVNN = 17,284   CVNN = 8,836   ratio = 0.51

────────────────────────────────────────────────────────────
  Dataset: Larmor
────────────────────────────────────────────────────────────
  [data] Generating 2000 Larmor precession snapshots...
  Training RVNN (17,284 params, MSE loss)...
  Training CVNN (8,836 params, fidelity loss)...
  Metric                             RVNN       CVNN
  ────────────────────────────────────────────────
  Fidelity (normalized)          0.999883   0.998733
  Norm (mean +/- std)           0.999+/-0.018  1.000+/-0.000
  Phase Error (rad)                0.0100     0.0418

────────────────────────────────────────────────────────────
  Dataset: Damped
────────────────────────────────────────────────────────────
  [data] Generating 2000 damped precession snapshots...
  Training RVNN (17,284 params, MSE loss)...
  Training CVNN (8,836 params, fidelity loss)...

In [6]:
print(f"\n{'='*70}")
print("  GENERATING VISUALIZATIONS")
print(f"{'='*70}")

short = {"Larmor": "Larmor", "Damped": "Damped", "Two-Qubit": "Two-Qubit"}

# ── FIG 1: Training Convergence ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for i, ds in enumerate(DATASETS):
    ax = axes[i]
    L = all_losses[ds]
    ep = np.arange(len(L['rvnn'])) * 10
    ax.semilogy(ep, L['rvnn'], color=C_RVNN, lw=1.5, alpha=0.8, label='RVNN (MSE)')
    ax.semilogy(ep, L['cvnn'], color=C_CVNN, lw=1.5, alpha=0.8, label='CVNN (Fidelity)')
    ax.set_title(short[ds], fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(framealpha=0.9)
    if i == 0: ax.set_ylabel('Loss (log)')
fig.suptitle('Training Convergence: RVNN vs Constrained CVNN',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/01_convergence.png"); plt.close()
print("  Saved 01_convergence.png")

# ── FIG 2: Fidelity Bar Chart ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(3); w = 0.3
fr = [all_results[d]['fid_rvnn'] for d in DATASETS]
fc = [all_results[d]['fid_cvnn'] for d in DATASETS]
b1 = ax.bar(x - w/2, fr, w, label='RVNN', color=C_RVNN, edgecolor='white', zorder=3)
b2 = ax.bar(x + w/2, fc, w, label='CVNN (constrained)', color=C_CVNN, edgecolor='white', zorder=3)
ax.axhline(1.0, color=C_IDEAL, ls='--', alpha=.6, lw=1.5, label='Ideal (F=1)')
ax.set_ylim(0.90, 1.005); ax.set_xticks(x)
ax.set_xticklabels(list(short.values()), fontsize=12)
ax.set_ylabel('Mean Quantum Fidelity')
ax.set_title('Quantum Fidelity Across Datasets', fontsize=14, fontweight='bold')
for b in b1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+.001,
            f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=9, color=C_RVNN)
for b in b2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+.001,
            f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=9, color=C_CVNN)
ax.legend(loc='lower left', framealpha=.9)
fig.savefig(f"{OUTPUT_DIR}/02_fidelity_bars.png"); plt.close()
print("  Saved 02_fidelity_bars.png")

# ── FIG 3: Norm Preservation ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for i, ds in enumerate(DATASETS):
    ax = axes[i]; P = all_preds[ds]; t = P['X'].flatten()
    nr = compute_norm(P['rvnn_raw']); nc = compute_norm(P['cvnn'])
    ax.plot(t, nr, color=C_RVNN, lw=1, alpha=.7, label='RVNN (raw)')
    ax.plot(t, nc, color=C_CVNN, lw=1.5, alpha=.8, label='CVNN')
    ax.axhline(1.0, color=C_IDEAL, ls='--', lw=1.5, alpha=.6, label='||ψ||=1')
    ax.fill_between(t, 1.0, nr, alpha=.1, color=C_RVNN)
    ax.set_title(short[ds], fontweight='bold'); ax.set_xlabel('Time (t)')
    if i == 0: ax.set_ylabel('||ψ_pred||')
    ax.legend(fontsize=8)
fig.suptitle('Norm Conservation: CVNN Enforces ||ψ||=1 by Construction',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/03_norm.png"); plt.close()
print("  Saved 03_norm.png")

# ── FIG 4: Phase Error ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for i, ds in enumerate(DATASETS):
    ax = axes[i]; R = all_results[ds]; t = all_preds[ds]['X'].flatten()
    ax.plot(t, R['phase_r_s'], color=C_RVNN, lw=1, alpha=.6,
            label=f"RVNN ({R['phase_r']:.4f} rad)")
    ax.plot(t, R['phase_c_s'], color=C_CVNN, lw=1.5, alpha=.8,
            label=f"CVNN ({R['phase_c']:.4f} rad)")
    ax.axhline(0, color=C_IDEAL, ls='--', alpha=.4)
    ax.set_title(short[ds], fontweight='bold'); ax.set_xlabel('Time (t)')
    if i == 0: ax.set_ylabel('Phase Error (rad)')
    ax.legend(fontsize=8)
fig.suptitle('Phase Accuracy: CVNN Preserves Quantum Phase',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/04_phase.png"); plt.close()
print("  Saved 04_phase.png")

# ── FIG 5: Fidelity Time Series (Larmor) ─────────────────────────────────────
fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 8),
                              gridspec_kw={'height_ratios': [2, 1]})
ds = 'Larmor'; t = all_preds[ds]['X'].flatten()
fr_s = all_results[ds]['fid_rvnn_s']; fc_s = all_results[ds]['fid_cvnn_s']
a1.plot(t, fr_s, color=C_RVNN, lw=1, alpha=.6, label=f"RVNN ({np.mean(fr_s):.4f})")
a1.plot(t, fc_s, color=C_CVNN, lw=2, alpha=.9, label=f"CVNN ({np.mean(fc_s):.4f})")
a1.axhline(1.0, color='k', ls='--', alpha=.3, label='Ideal')
a1.fill_between(t, fr_s, 1.0, alpha=.08, color=C_RVNN)
a1.set_ylabel('Fidelity  F = |⟨ψ|φ⟩|²')
a1.set_title('Fidelity vs Time — Larmor Precession', fontsize=14, fontweight='bold')
a1.legend(fontsize=11)
a1.set_ylim(min(np.min(fr_s), np.min(fc_s)) - 0.01, 1.005)
a2.fill_between(t, 0, np.abs(1 - fr_s), alpha=.3, color=C_RVNN, label='RVNN |1-F|')
a2.fill_between(t, 0, np.abs(1 - fc_s), alpha=.3, color=C_CVNN, label='CVNN |1-F|')
a2.set_ylabel('|1 - F|'); a2.set_xlabel('Time (t)'); a2.legend()
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/05_fidelity_ts.png"); plt.close()
print("  Saved 05_fidelity_ts.png")

# ── FIG 6: Statistical Box Plots ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ds in enumerate(DATASETS):
    ax = axes[i]; S = stat_results[ds]
    bp = ax.boxplot([S['rvnn'], S['cvnn']], labels=['RVNN', 'CVNN'],
                    patch_artist=True, widths=.5,
                    medianprops=dict(color='navy', lw=2),
                    whiskerprops=dict(lw=1.5), capprops=dict(lw=1.5))
    bp['boxes'][0].set(facecolor=C_RVNN, alpha=.3)
    bp['boxes'][1].set(facecolor=C_CVNN, alpha=.3)
    for j, d in enumerate([S['rvnn'], S['cvnn']]):
        ax.scatter([j+1]*len(d), d, alpha=.6, s=30, zorder=5,
                   color=[C_RVNN, C_CVNN][j])
    sig = '***' if S['p']<.001 else '**' if S['p']<.01 else '*' if S['p']<.05 else 'n.s.'
    ax.set_title(f"{short[ds]}\np = {S['p']:.2e} ({sig})", fontweight='bold')
    if i == 0: ax.set_ylabel('Mean Fidelity')
fig.suptitle(f'Fidelity Distribution ({N_TRIALS} Trials)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/06_stats_box.png"); plt.close()
print("  Saved 06_stats_box.png")

# ── FIG 7: Extrapolation ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
te = X_ext.flatten()
ax.plot(te, fid_r_ext, color=C_RVNN, lw=1.5, alpha=.7, label='RVNN')
ax.plot(te, fid_c_ext, color=C_CVNN, lw=2, label='CVNN')
ax.axvline(X_ext[split, 0], color=C_IDEAL, ls='--', lw=2, label='Train/Test boundary')
ax.axhline(1.0, color='k', ls=':', alpha=.3)
ax.fill_betweenx([0, 1.05], X_ext[split, 0], X_ext[-1, 0], alpha=.06, color=C_IDEAL)
ax.annotate('Training', xy=(4, .55), fontsize=12, ha='center', color='gray')
ax.annotate('Extrapolation', xy=(10, .55), fontsize=12, ha='center', color=C_IDEAL)
ax.set_title(f'Extrapolation — RVNN: {np.mean(fid_r_ext[split:]):.3f}  '
             f'CVNN: {np.mean(fid_c_ext[split:]):.3f}',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Time (t)'); ax.set_ylabel('Fidelity')
ax.set_ylim(0.0, 1.05); ax.legend(fontsize=11)
fig.savefig(f"{OUTPUT_DIR}/07_extrapolation.png"); plt.close()
print("  Saved 07_extrapolation.png")

# ── FIG 8: Component Predictions (Larmor) ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
ds = 'Larmor'; P = all_preds[ds]; t = P['X'].flatten()
comps = [(0, 'real', 'Re(α)'), (0, 'imag', 'Im(α)'),
         (1, 'real', 'Re(β)'), (1, 'imag', 'Im(β)')]
for idx, (c, part, label) in enumerate(comps):
    ax = axes[idx//2][idx%2]
    tv = getattr(P['y'][:, c], part)
    rv = getattr(P['rvnn'][:, c], part)
    cv = getattr(P['cvnn'][:, c], part)
    ax.plot(t, tv, 'k-', lw=2, alpha=.4, label='True')
    ax.plot(t, rv, color=C_RVNN, lw=1, ls='--', alpha=.7, label='RVNN')
    ax.plot(t, cv, color=C_CVNN, lw=1.5, ls='-.', alpha=.8, label='CVNN')
    ax.set_title(label, fontweight='bold', fontsize=13)
    ax.set_xlabel('Time (t)'); ax.legend(fontsize=9)
fig.suptitle('Component Predictions — Larmor', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/08_components.png"); plt.close()
print("  Saved 08_components.png")

# ── FIG 9: Bloch Sphere ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 5))
def bloch(ax, psi, title, color):
    theta = 2 * np.arccos(np.clip(np.abs(psi[:, 0]), 0, 1))
    phi   = np.angle(psi[:, 1]) - np.angle(psi[:, 0])
    bx, by, bz = np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)
    u, v = np.linspace(0, 2*np.pi, 30), np.linspace(0, np.pi, 20)
    ax.plot_surface(np.outer(np.cos(u), np.sin(v)),
                    np.outer(np.sin(u), np.sin(v)),
                    np.outer(np.ones_like(u), np.cos(v)), alpha=.05, color='gray')
    ax.plot(bx, by, bz, color=color, lw=1.5, alpha=.7)
    ax.scatter([bx[0]], [by[0]], [bz[0]], c='green', s=50, zorder=5)
    ax.scatter([bx[-1]], [by[-1]], [bz[-1]], c='red', s=50, zorder=5)
    ax.set(xlabel='X', ylabel='Y', zlabel='Z', title=title,
           xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), zlim=(-1.1, 1.1))

P = all_preds['Larmor']
for i, (data, title, col) in enumerate([
        (P['y'],    'True State', 'black'),
        (P['rvnn'], 'RVNN',      C_RVNN),
        (P['cvnn'], 'CVNN',      C_CVNN)]):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    bloch(ax, data, title, col)
fig.suptitle('Bloch Sphere — Larmor Precession', fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/09_bloch.png"); plt.close()
print("  Saved 09_bloch.png")

# ── FIG 10: Summary Dashboard ────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, hspace=.35, wspace=.3)

# (a) Fidelity
ax = fig.add_subplot(gs[0, 0]); x = np.arange(3)
fr_v = [all_results[d]['fid_rvnn'] for d in DATASETS]
fc_v = [all_results[d]['fid_cvnn'] for d in DATASETS]
ax.bar(x-.15, fr_v, .3, color=C_RVNN, alpha=.8, label='RVNN')
ax.bar(x+.15, fc_v, .3, color=C_CVNN, alpha=.8, label='CVNN')
ax.set_xticks(x); ax.set_xticklabels(list(short.values()), fontsize=9)
ax.axhline(1, color='k', ls=':', alpha=.3); ax.set_ylim(.9, 1.005)
ax.set_ylabel('Fidelity'); ax.set_title('(a) Fidelity', fontweight='bold')
ax.legend(fontsize=8)

# (b) Norm deviation
ax = fig.add_subplot(gs[0, 1])
nd_r = [abs(all_results[d]['norm_r_mean'] - 1) for d in DATASETS]
nd_c = [abs(all_results[d]['norm_c_mean'] - 1) for d in DATASETS]
ax.bar(x-.15, nd_r, .3, color=C_RVNN, alpha=.8, label='RVNN')
ax.bar(x+.15, nd_c, .3, color=C_CVNN, alpha=.8, label='CVNN')
ax.set_xticks(x); ax.set_xticklabels(list(short.values()), fontsize=9)
ax.set_ylabel('|‖ψ‖ - 1|'); ax.set_title('(b) Norm Deviation', fontweight='bold')
ax.legend(fontsize=8)

# (c) Phase error
ax = fig.add_subplot(gs[0, 2])
pe_r = [all_results[d]['phase_r'] for d in DATASETS]
pe_c = [all_results[d]['phase_c'] for d in DATASETS]
ax.bar(x-.15, pe_r, .3, color=C_RVNN, alpha=.8, label='RVNN')
ax.bar(x+.15, pe_c, .3, color=C_CVNN, alpha=.8, label='CVNN')
ax.set_xticks(x); ax.set_xticklabels(list(short.values()), fontsize=9)
ax.set_ylabel('Phase Error (rad)'); ax.set_title('(c) Phase Error', fontweight='bold')
ax.legend(fontsize=8)

# (d) Stats box (Larmor)
ax = fig.add_subplot(gs[1, 0]); S = stat_results['Larmor']
bp = ax.boxplot([S['rvnn'], S['cvnn']], labels=['RVNN', 'CVNN'],
                patch_artist=True, widths=.4, medianprops=dict(color='navy', lw=2))
bp['boxes'][0].set(facecolor=C_RVNN, alpha=.3)
bp['boxes'][1].set(facecolor=C_CVNN, alpha=.3)
ax.set_title(f"(d) Stats  p={S['p']:.2e}", fontweight='bold')
ax.set_ylabel('Fidelity')

# (e) Convergence (Larmor)
ax = fig.add_subplot(gs[1, 1]); L = all_losses['Larmor']
ep = np.arange(len(L['rvnn'])) * 10
ax.semilogy(ep, L['rvnn'], color=C_RVNN, alpha=.8, label='RVNN')
ax.semilogy(ep, L['cvnn'], color=C_CVNN, alpha=.8, label='CVNN')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('(e) Convergence', fontweight='bold'); ax.legend(fontsize=8)

# (f) Params
ax = fig.add_subplot(gs[1, 2])
bars = ax.bar(['RVNN\n(128 hidden)', 'CVNN\n(64 complex)'],
              [n_rvnn, n_cvnn], color=[C_RVNN, C_CVNN], alpha=.8, width=.4)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+200,
            f'{int(b.get_height()):,}', ha='center', fontsize=10)
ax.set_ylabel('Trainable Parameters')
ax.set_title('(f) Model Size', fontweight='bold')

fig.suptitle('Summary Dashboard — CVNN vs RVNN',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/10_dashboard.png"); plt.close()
print("  Saved 10_dashboard.png")

# ── FIG 11: Fair Parameter-Matched Comparison ────────────────────────────────
C_FAIR = '#9B59B6'   # purple for FairRVNN
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ds in enumerate(DATASETS):
    ax = axes[i]; S = fair_stat_results[ds]
    bp = ax.boxplot([S['fair'], S['cvnn']], labels=['FairRVNN', 'CVNN'],
                    patch_artist=True, widths=.5,
                    medianprops=dict(color='navy', lw=2),
                    whiskerprops=dict(lw=1.5), capprops=dict(lw=1.5))
    bp['boxes'][0].set(facecolor=C_FAIR, alpha=.3)
    bp['boxes'][1].set(facecolor=C_CVNN, alpha=.3)
    for j, d in enumerate([S['fair'], S['cvnn']]):
        ax.scatter([j+1]*len(d), d, alpha=.6, s=30, zorder=5,
                   color=[C_FAIR, C_CVNN][j])
    sig = '***' if S['p']<.001 else '**' if S['p']<.01 else '*' if S['p']<.05 else 'n.s.'
    FR = fair_results[ds]
    ax.set_title(f"{short[ds]}\nFairRVNN={FR['n_fair']:,} vs CVNN={FR['n_cvnn']:,}\n"
                 f"p = {S['p']:.2e} ({sig})", fontweight='bold', fontsize=10)
    if i == 0: ax.set_ylabel('Mean Fidelity')
fig.suptitle(f'Parameter-Matched Comparison ({N_TRIALS} Trials)\n'
             'Same parameter budget — does complex arithmetic help?',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/11_fair_comparison.png"); plt.close()
print("  Saved 11_fair_comparison.png")

# ── FIG 12: Physical Validity ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(3); w = 0.3
vr = [validity_results[d]['rvnn'] for d in DATASETS]
vc = [validity_results[d]['cvnn'] for d in DATASETS]
b1 = ax.bar(x - w/2, vr, w, label='RVNN (raw)', color=C_RVNN, edgecolor='white', zorder=3)
b2 = ax.bar(x + w/2, vc, w, label='CVNN', color=C_CVNN, edgecolor='white', zorder=3)
ax.axhline(100.0, color=C_IDEAL, ls='--', alpha=.6, lw=1.5, label='Ideal (100%)')
ax.set_ylim(0, 115); ax.set_xticks(x)
ax.set_xticklabels(list(short.values()), fontsize=12)
ax.set_ylabel('Valid Predictions (%)')
ax.set_title('Physical Validity: Born Rule Compliance\n'
             '% of outputs with ||ψ|| ∈ [0.99, 1.01]', fontsize=14, fontweight='bold')
for b in b1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1.5,
            f'{b.get_height():.1f}%', ha='center', va='bottom', fontsize=10, color=C_RVNN)
for b in b2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1.5,
            f'{b.get_height():.1f}%', ha='center', va='bottom', fontsize=10, color=C_CVNN)
ax.legend(loc='lower left', framealpha=.9)
fig.savefig(f"{OUTPUT_DIR}/12_validity.png"); plt.close()
print("  Saved 12_validity.png")

# ── FIG 13: Noise Robustness ─────────────────────────────────────────────────
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.5))
sigma = noise_results['sigma']
a1.plot(sigma, noise_results['rvnn'], 'o-', color=C_RVNN, lw=2, ms=6,
        label=f"RVNN ({n_rvnn:,} params)")
a1.plot(sigma, noise_results['fair'], 's-', color=C_FAIR, lw=2, ms=6,
        label=f"FairRVNN (~{fair_results['Larmor']['n_fair']:,} params)")
a1.plot(sigma, noise_results['cvnn'], 'D-', color=C_CVNN, lw=2, ms=6,
        label=f"CVNN ({n_cvnn:,} params)")
a1.axhline(1.0, color='k', ls=':', alpha=.3)
a1.set_xlabel('Input Noise Level (σ)', fontsize=12)
a1.set_ylabel('Mean Fidelity', fontsize=12)
a1.set_title('Fidelity vs Input Noise', fontweight='bold', fontsize=13)
a1.set_ylim(0.0, 1.05)
a1.legend(fontsize=10)
a2.plot(sigma, noise_results['norm_rvnn'], 'o-', color=C_RVNN, lw=2, ms=6,
        label='RVNN σ(||ψ||)')
a2.plot(sigma, noise_results['norm_fair'], 's-', color=C_FAIR, lw=2, ms=6,
        label='FairRVNN σ(||ψ||)')
a2.axhline(0.0, color=C_CVNN, ls='--', lw=2, alpha=.8, label='CVNN σ(||ψ||) = 0 (always)')
a2.set_xlabel('Input Noise Level (σ)', fontsize=12)
a2.set_ylabel('Norm Std Dev', fontsize=12)
a2.set_title('Norm Stability vs Input Noise', fontweight='bold', fontsize=13)
a2.legend(fontsize=10)
fig.suptitle('Noise Robustness — Larmor Precession', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); fig.savefig(f"{OUTPUT_DIR}/13_noise_robustness.png"); plt.close()
print("  Saved 13_noise_robustness.png")

# ── Final Summary Table ──────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("  FINAL SUMMARY")
print(f"{'='*70}")

print("\n  ── Original Comparison (RVNN 17K vs CVNN 9K) ──")
rows = []
for ds in DATASETS:
    R = all_results[ds]; S = stat_results[ds]
    rows.append({
        'Dataset': ds,
        'RVNN Fidelity': f"{R['fid_rvnn']:.4f}",
        'CVNN Fidelity': f"{R['fid_cvnn']:.4f}",
        'RVNN Phase Err': f"{R['phase_r']:.4f}",
        'CVNN Phase Err': f"{R['phase_c']:.4f}",
        'p-value': f"{S['p']:.2e}",
    })
print(pd.DataFrame(rows).set_index('Dataset').to_string())

print("\n  ── Fair Comparison (parameter-matched) ──")
rows2 = []
for ds in DATASETS:
    FR = fair_results[ds]; FS = fair_stat_results[ds]
    sig = "***" if FS['p']<.001 else "**" if FS['p']<.01 else "*" if FS['p']<.05 else "n.s."
    rows2.append({
        'Dataset': ds,
        'FairRVNN Fid': f"{FR['fid_fair']:.4f}",
        'CVNN Fid': f"{FR['fid_cvnn']:.4f}",
        'FairRVNN Params': FR['n_fair'],
        'CVNN Params': FR['n_cvnn'],
        'p-value': f"{FS['p']:.2e} ({sig})",
    })
print(pd.DataFrame(rows2).set_index('Dataset').to_string())

print("\n  ── Physical Validity (% Born-rule compliant) ──")
for ds in DATASETS:
    V = validity_results[ds]
    print(f"  {ds:<12}  RVNN: {V['rvnn']:6.1f}%   CVNN: {V['cvnn']:6.1f}%")

print(f"\n  All {len(os.listdir(OUTPUT_DIR))} figures saved to ./{OUTPUT_DIR}/")
print("  Study Complete.")


  GENERATING VISUALIZATIONS
  Saved 01_convergence.png
  Saved 02_fidelity_bars.png
  Saved 03_norm.png
  Saved 04_phase.png
  Saved 05_fidelity_ts.png
  Saved 06_stats_box.png
  Saved 07_extrapolation.png
  Saved 08_components.png
  Saved 09_bloch.png
  Saved 10_dashboard.png
  Saved 11_fair_comparison.png
  Saved 12_validity.png
  Saved 13_noise_robustness.png

  FINAL SUMMARY

  ── Original Comparison (RVNN 17K vs CVNN 9K) ──
          RVNN Fidelity CVNN Fidelity RVNN Phase Err CVNN Phase Err   p-value
Dataset                                                                      
Larmor           0.9999        0.9987         0.0100         0.0418  2.66e-01
Damped           0.9999        0.9999         0.0172         0.0179  2.11e-01
Two-Qubit        0.9989        0.9978         0.0528         0.0905  7.40e-02

  ── Fair Comparison (parameter-matched) ──
          FairRVNN Fid CVNN Fid  FairRVNN Params  CVNN Params          p-value
Dataset                                             